# Exercise: Sentiment guesser

## Understanding top-down & bottom-up approaches

Learning objectives
- Understand the difference between top-down (rule-based) and bottom-up (pattern-based) approaches in sentiment analysis.
- Implement a simple sentiment detection model using both approaches.
- Reflect on the advantages and limitations of each method.


In [ ]:
# Note that this exercise requires the following packages: 
#!pip3 install wordcloud matplotlib pandas numpy vaderSentiment seaborn

In [ ]:
## load packages
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud

In [ ]:
GROUP_NAME = "GROUP AWESOME" # change this to your group name

In [ ]:
data = {"review": [
    "I absolutely love this product! It works great and is amazing.",
    "This is the worst purchase I have ever made. Totally terrible!",
    "It's okay, but I expected better. Not great but not bad either.",
    "Fantastic quality! Exceeded my expectations.",
    "I hate this so much. Worst thing ever!",
    "Pretty decent, could be improved but overall not bad.",
    "Superb experience! Highly recommend.",
    "Disappointed. Not what I expected at all.",
    "Well, it works... I guess.",  
    "I was excited to try this, but it completely failed my expectations.",  
    "It's a product."  
]}

df = pd.DataFrame(data)
df

### --- Step 1: Manual Sentiment Classification ---

With your team, please discuss whether you should enter _Positive_, _Negative_, or _Neutral_ for each review.
Please also rate your certainty for the label from 1 (very uncertain) to 5 (very certain).

In [ ]:
df["ManualSentiment"] = ["" for _ in range(len(df))]  # Placeholder for student input

print("\n### Manual Sentiment Classification ###")
print("Please manually classify the following reviews as Positive, Negative, or Neutral.\n")

valid_sentiments = {"Positive", "Negative", "Neutral"}

for i, review in enumerate(df["review"]):
    while True:
        sentiment = input(
            f"Review: {review}\nYour Sentiment (Positive/Negative/Neutral): "
        ).strip().capitalize()
        
        if sentiment in valid_sentiments:
            df.at[i, "ManualSentiment"] = sentiment
            break
        
        print("Invalid input. Please enter Positive, Negative, or Neutral.")
        

In [ ]:
df

# --- Step 2: Top-Down Approach ---
Use a predefined lexicon of positive and negative words. We make the following rule: 
- Positive Sentiment: If the number of positive words in the review exceeds the number of negative words, the sentiment is labeled as Positive.
- Negative Sentiment: If the number of negative words exceeds the number of positive words, the sentiment is labeled as Negative.
- Neutral Sentiment: If the number of positive words equals the number of negative words (or neither exceeds the other), the sentiment is labeled as Neutral.


In [ ]:
positive_words = ["love", "great", "excellent", "amazing", "happy", "fantastic", "superb", "recommend"]
negative_words = ["worst", "terrible", "awful", "hate", "bad", "disappointed"]

def top_down_sentiment(text):
    text = text.lower()
    
    pos_count = 0
    neg_count = 0
    
    for word in text.split():
        if word in positive_words:
            pos_count += 1
        elif word in negative_words:
            neg_count += 1

    if pos_count > neg_count:
        return "Positive"
    elif neg_count > pos_count:
        return "Negative"
    else:
        return "Neutral"

df["TopDownSentiment"] = df["review"].apply(top_down_sentiment)

df

Can *you* think of additional words that should be added to the list? how does adding more words change your results?


# --- Step 3: Bottom-up approach ---
# Use word patterns in the data to understand sentiment


In this step, we move from manually labeling reviews to analyzing patterns in the text.

What does the code do?

1. We split the reviews into two groups:
   - Positive reviews
   - Negative reviews

2. For each group, we:
   - Combine all reviews into one text
   - Split the text into individual words
   - Count how often each word appears

3. We then visualize the most common words using word clouds.

The idea:
- Positive reviews will contain more positive words (e.g., "great", "amazing")
- Negative reviews will contain more negative words (e.g., "bad", "terrible")

This helps us see how language differs between sentiments.

👉 Important:
This is called a "bottom-up" approach because we let patterns in the data
guide our understanding, instead of defining rules in advance.


In [ ]:
from collections import Counter
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Function to get word counts from reviews
def get_word_counts(reviews):
    words = " ".join(reviews).lower().split()
    return Counter(words)

# Split reviews into positive and negative groups
positive_reviews = df[df["ManualSentiment"] == "Positive"]["review"]
negative_reviews = df[df["ManualSentiment"] == "Negative"]["review"]

# Get word frequencies
positive_counts = get_word_counts(positive_reviews)
negative_counts = get_word_counts(negative_reviews)

# Function to generate a word cloud
def generate_wordcloud(word_counts, title):
    wordcloud = WordCloud(width=800, height=400, background_color="white") \
        .generate_from_frequencies(word_counts)
    
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.title(title)
    plt.show()

# Display word clouds
generate_wordcloud(positive_counts, "Positive Reviews")
generate_wordcloud(negative_counts, "Negative Reviews")


In [ ]:
df

# --- Step 4: VADER Sentiment Analysis ---

In the final step, we will use a pre-trained model for sentiment classification. Since VADER relies on a predefined lexicon with sentiment scores assigned in advance, you could argue that it follows a top-down approach—applying predefined knowledge (the dictionary) to analyze new text. However, it also exhibits bottom-up characteristics because it computes the overall sentiment by aggregating individual word scores and adjusting based on syntactic rules (e.g., negation, intensifiers, punctuation).

So, it’s a bit of both:
- Top-down because it starts with a pre-built sentiment dictionary.
- Bottom-up because it builds sentiment from individual words and adjusts based on context.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = analyzer.polarity_scores(text)
    return "Positive" if score["compound"] > 0.05 else "Negative" if score["compound"] < -0.05 else "Neutral"

df["VADER_Sentiment"] = df["review"].apply(vader_sentiment)

In [ ]:
df

# --- Step 5: Agreement calculation  ---

What does this tell us about agreement with human annotations?

In [ ]:
def agreement_score(manual, predicted):
    return 1 if manual == predicted else 0

# apply the agreement score calculation
df["agreement_topdown"] = df.apply(lambda row: agreement_score(row["ManualSentiment"], row["TopDownSentiment"]), axis=1)
df["agreement_vader"] = df.apply(lambda row: agreement_score(row["ManualSentiment"], row["VADER_Sentiment"]), axis=1)

# calculate non-weighted agreement percentages
def non_weighted_agreement(agreement_col):
    return df[agreement_col].mean()

agreement_summary = {
    "top-down agreement": non_weighted_agreement("agreement_topdown"),
    "vader agreement": non_weighted_agreement("agreement_vader")
}

# --- Step 6: visualization of agreement scores ---

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

agreement_df = pd.DataFrame(
    list(agreement_summary.items()),
    columns=["SentimentMethod", "AgreementScore"]
)

# Plot agreement scores using Seaborn
plt.figure(figsize=(8, 5))

sns.barplot(
    x="SentimentMethod",
    y="AgreementScore",
    hue="SentimentMethod",  
    data=agreement_df,
    palette="Set2",
    legend=False             
)

# Set plot labels and title
plt.ylim(0, 1)
plt.xlabel("Sentiment analysis method")
plt.ylabel("Agreement score")
plt.title(f"Agreement between manual annotations and automated methods {GROUP_NAME}")

# Save the figure
plt.savefig(f"agreement_scores_{GROUP_NAME}.png")

# Show the plot
plt.show()
